# 05 — Bayesian models and results

## Manuscript crosswalk

- **Methods:** unified Bayesian multilevel models; priors; sampling; posterior contrasts; diagnostics.
- **Results:** context-specific stage effects (Δ), stage-by-context contrasts (Ψ), metric-specific estimates, and the two-pair common-support sensitivity analysis.
- **Figure:** Figure 3 and posterior-predictive/trace diagnostics.

This notebook is report-only. It reads checksum-verified model tables, posterior draws, summaries, and diagnostics. It never compiles Stan code or starts MCMC sampling.

## Model specification

One Gaussian identity-link model is fitted for sequence structure and one for call structure:

```r
standardized_distance ~ stage * context * metric +
  (1 + stage + context + stage:context || pair_id) +
  (1 | mm(repertoire_a_id, repertoire_b_id))
```

Stage is centered as before = −0.5 and after = +0.5. Population-level coefficients and the intercept have Normal(0, 1) priors; group-level and residual standard deviations have Exponential(1) priors. Each fit uses four chains of 4,000 iterations, including 2,000 warm-up iterations.

For each posterior draw, Δ is expected standardized distance after minus before within context. The direct audience contrast is Ψ = Δ\(_\mathrm{Partner}\) − Δ\(_\mathrm{Non-partner}\). Negative Δ means convergence; positive Ψ means that the reduction is greater in the Non-partner context. Full details are in [the canonical analysis specification](../docs/analysis_specification.md).

In [ ]:
from __future__ import annotations

import hashlib
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter inside a clone containing README.md and data/."
    )


ROOT = find_repo_root()
EXPENSIVE_STEPS = {
    "recompute_sequence": False,
    "recompute_dtw": False,
    "retrain_vae": False,
    "refit_models": False,
}
if any(EXPENSIVE_STEPS.values()):
    raise RuntimeError(
        "This report notebook cannot fit models. Use "
        "'bash scripts/run_repro_pipeline.sh --mode full --refit-models' explicitly."
    )

pd.DataFrame({"step": EXPENSIVE_STEPS.keys(), "enabled": EXPENSIVE_STEPS.values()})

In [ ]:
MANIFEST_PATHS = (
    ROOT / "data/processed/manifest.json",
    ROOT / "data/cache/manifest.json",
    ROOT / "data/derived/manifest.json",
    ROOT / "results/manifest.json",
    ROOT / "results/model_manifest.json",
)
FIGURE_PROVENANCE_PATH = ROOT / "results/figures/figure_provenance.csv"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_manifests() -> list[tuple[Path, dict]]:
    manifests = []
    for path in MANIFEST_PATHS:
        if path.is_file():
            with path.open(encoding="utf-8") as handle:
                manifests.append((path, json.load(handle)))
    if not manifests:
        raise FileNotFoundError("No provenance manifest was found.")
    return manifests


def iter_records(payload: dict):
    for section_name in ("artifacts", "sources", "inputs", "files"):
        section = payload.get(section_name, {})
        if isinstance(section, dict):
            for key, value in section.items():
                record = value if isinstance(value, dict) else {"sha256": value}
                yield str(record.get("path", key)), record
        elif isinstance(section, list):
            for record in section:
                if isinstance(record, dict) and record.get("path"):
                    yield str(record["path"]), record


def record_sha256(record: dict) -> str | None:
    value = record.get("sha256") or record.get("checksum_sha256")
    if value:
        return str(value).removeprefix("sha256:")
    checksum = record.get("checksum")
    if isinstance(checksum, str):
        return checksum.removeprefix("sha256:")
    if isinstance(checksum, dict) and checksum.get("algorithm", "").lower() == "sha256":
        return checksum.get("value")
    return None


MANIFESTS = load_manifests()
FIGURE_PROVENANCE = (
    pd.read_csv(FIGURE_PROVENANCE_PATH) if FIGURE_PROVENANCE_PATH.is_file() else pd.DataFrame()
)


def registered_artifact(relative_path: str) -> Path:
    path = ROOT / relative_path
    if not path.is_file():
        raise FileNotFoundError(
            f"Required result is missing: {relative_path}. Cached mode will not fit a replacement model."
        )
    normalized = Path(relative_path).as_posix()
    for manifest_path, payload in MANIFESTS:
        for recorded_path, record in iter_records(payload):
            recorded = Path(recorded_path)
            same_path = (recorded.resolve() == path.resolve()) if recorded.is_absolute() else (recorded.as_posix().lstrip("./") == normalized)
            if same_path:
                expected = record_sha256(record)
                if not expected:
                    raise RuntimeError(f"No SHA-256 for {relative_path} in {manifest_path}.")
                if sha256_file(path).lower() != expected.lower():
                    raise RuntimeError(f"Checksum mismatch for {relative_path}.")
                return path
    if not FIGURE_PROVENANCE.empty and {"path", "sha256"}.issubset(FIGURE_PROVENANCE.columns):
        match = FIGURE_PROVENANCE.loc[FIGURE_PROVENANCE["path"] == normalized]
        if len(match) == 1 and sha256_file(path).lower() == str(match.iloc[0]["sha256"]).lower():
            return path
    raise RuntimeError(f"{relative_path} is not registered in a provenance manifest.")

## Load saved posterior expected values

Each row contains one draw for one focal pair, audience context, and metric, together with expected before and after distances and their difference. The `common_support_pair` field marks the two pairs observed in every stage-by-context cell.

In [ ]:
# Use the R model layer as the single authority for cache provenance. This
# command validates only; it cannot compile Stan or start sampling.
try:
    cache_audit = subprocess.run(
        ["Rscript", str(ROOT / "R/run_models.R"), "--validate-cache-only"],
        cwd=ROOT,
        text=True,
        capture_output=True,
    )
except FileNotFoundError as error:
    raise RuntimeError("Rscript is required for the report-only posterior provenance audit.") from error
if cache_audit.returncode != 0:
    raise RuntimeError(
        "The R provenance audit rejected the posterior cache. No model will be refitted.\n"
        + cache_audit.stderr
    )
for sidecar in (
    "results/posterior_draws/sequence_stage_effect_draws.csv.gz.metadata.json",
    "results/posterior_draws/call_stage_effect_draws.csv.gz.metadata.json",
):
    registered_artifact(sidecar)

sequence_model = pd.read_csv(registered_artifact("data/derived/model_sequence.csv"))
call_model = pd.read_csv(registered_artifact("data/derived/model_call.csv"))
sequence_draws = pd.read_csv(
    registered_artifact("results/posterior_draws/sequence_stage_effect_draws.csv.gz")
)
call_draws = pd.read_csv(
    registered_artifact("results/posterior_draws/call_stage_effect_draws.csv.gz")
)

DRAW_COLUMNS = {
    ".draw", "analysis", "pair", "context", "metric", "metric_label",
    "expected_before", "expected_after", "delta", "common_support_pair",
}


def normalize_analysis(value: object) -> str:
    value = str(value).strip().lower().replace("_", " ")
    aliases = {"sequence": "sequence", "sequence structure": "sequence",
               "call": "call", "call structure": "call",
               "acoustic": "call", "spectral": "call"}
    return aliases.get(value, value)
for name, table, expected_analysis in (
    ("sequence draws", sequence_draws, "sequence"),
    ("call draws", call_draws, "call"),
):
    if missing := sorted(DRAW_COLUMNS.difference(table.columns)):
        raise ValueError(f"{name} is missing canonical columns: {missing}")
    if set(table["analysis"].map(normalize_analysis)) != {expected_analysis}:
        raise ValueError(f"{name} has an unexpected analysis label.")
    numeric = table[["expected_before", "expected_after", "delta"]]
    if not np.isfinite(numeric.to_numpy()).all():
        raise ValueError(f"{name} contains non-finite posterior values.")
    if not np.allclose(table["expected_after"] - table["expected_before"], table["delta"]):
        raise ValueError(f"{name} contains inconsistent delta values.")
    if table[".draw"].nunique() != 8000:
        raise ValueError(f"{name} does not contain the expected 8,000 post-warm-up draws.")

if len(sequence_model) != 1324 or len(call_model) != 1724:
    raise AssertionError("Model-table row counts do not match the manuscript.")

{"sequence_draw_rows": len(sequence_draws), "call_draw_rows": len(call_draws),
 "sequence_draws": sequence_draws[".draw"].nunique(),
 "call_draws": call_draws[".draw"].nunique()}

## Reconstruct Δ and Ψ from the draws

Pair-specific expected distances are averaged equally across included pairs. Combined estimates then average the four standardized metric effects with equal weight. This weighting defines the reported contrasts; it does not alter how the individual comparison rows contributed to model fitting.

In [ ]:
def normalize_context(value: object) -> str:
    value = str(value).strip().lower().replace("_", "-")
    return "non-partner" if value in {"stranger", "nonpartner", "non-paired"} else value


def as_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series
    return series.astype(str).str.strip().str.lower().isin({"true", "t", "1", "yes"})


def posterior_contrasts(draws: pd.DataFrame, support: str) -> dict[str, pd.DataFrame]:
    work = draws.copy()
    work["context"] = work["context"].map(normalize_context)
    if set(work["context"]) != {"partner", "non-partner"}:
        raise ValueError(f"Unexpected context labels: {set(work['context'])}")
    if support == "common_support_A_B":
        work = work.loc[as_bool(work["common_support_pair"])].copy()
    elif support != "all_pairs":
        raise ValueError(f"Unknown support set: {support}")

    expected_pair_count = 2 if support == "common_support_A_B" else 3
    if work["pair"].nunique() != expected_pair_count:
        raise ValueError(
            f"{support} contains {work['pair'].nunique()} pairs; expected {expected_pair_count}."
        )

    by_metric = (
        work.groupby([".draw", "context", "metric", "metric_label"], observed=True, as_index=False)
        .agg(delta=("delta", "mean"))
    )
    combined = (
        by_metric.groupby([".draw", "context"], observed=True, as_index=False)
        .agg(delta=("delta", "mean"))
        .assign(metric="combined", metric_label="Combined")
    )
    metric_psi = (
        by_metric.pivot(index=[".draw", "metric", "metric_label"], columns="context", values="delta")
        .reset_index()
    )
    metric_psi["psi"] = metric_psi["partner"] - metric_psi["non-partner"]
    combined_psi = combined.pivot(index=".draw", columns="context", values="delta").reset_index()
    combined_psi["psi"] = combined_psi["partner"] - combined_psi["non-partner"]
    combined_psi["metric"] = "combined"
    combined_psi["metric_label"] = "Combined"
    return {
        "by_metric": by_metric.assign(support=support),
        "combined": combined.assign(support=support),
        "metric_psi": metric_psi.assign(support=support),
        "combined_psi": combined_psi.assign(support=support),
    }


def summarize_draws(table: pd.DataFrame, value: str, group: list[str]) -> pd.DataFrame:
    return (
        table.groupby(group, observed=True)[value]
        .agg(
            estimate_median="median",
            lower_95=lambda x: x.quantile(0.025),
            upper_95=lambda x: x.quantile(0.975),
            n_draws="size",
        )
        .reset_index()
    )


contrast_sets = {}
for analysis, draws in (("sequence", sequence_draws), ("call", call_draws)):
    for support in ("all_pairs", "common_support_A_B"):
        contrast_sets[(analysis, support)] = posterior_contrasts(draws, support)

metric_delta_summary = pd.concat([
    summarize_draws(parts["by_metric"], "delta", ["support", "context", "metric", "metric_label"])
    .assign(analysis=analysis)
    for (analysis, support), parts in contrast_sets.items()
], ignore_index=True)
combined_delta_summary = pd.concat([
    summarize_draws(parts["combined"], "delta", ["support", "context", "metric", "metric_label"])
    .assign(analysis=analysis)
    for (analysis, support), parts in contrast_sets.items()
], ignore_index=True)
metric_psi_summary = pd.concat([
    summarize_draws(parts["metric_psi"], "psi", ["support", "metric", "metric_label"])
    .assign(analysis=analysis)
    for (analysis, support), parts in contrast_sets.items()
], ignore_index=True)
combined_psi_summary = pd.concat([
    summarize_draws(parts["combined_psi"], "psi", ["support", "metric", "metric_label"])
    .assign(analysis=analysis)
    for (analysis, support), parts in contrast_sets.items()
], ignore_index=True)

display(metric_delta_summary.query("support == 'all_pairs'").round(3))
display(metric_psi_summary.query("support == 'all_pairs'").round(3))

## Manuscript headline validation

The expected medians and interval limits are non-computational targets transcribed from the manuscript. The observed values below are newly summarized from the checksum-verified draws. Agreement is assessed at the manuscript’s two-decimal reporting precision.

In [ ]:
def select_summary(table: pd.DataFrame, analysis: str, support: str, metric: str,
                   context: str | None = None) -> pd.Series:
    selected = table.loc[
        (table["analysis"] == analysis)
        & (table["support"] == support)
        & (table["metric"].astype(str).str.lower() == metric)
    ]
    if context is not None:
        selected = selected.loc[selected["context"] == context]
    if len(selected) != 1:
        raise ValueError(
            f"Expected one summary row for analysis={analysis}, support={support}, metric={metric}, context={context}; "
            f"found {len(selected)}."
        )
    return selected.iloc[0]


reference_targets = pd.read_csv(
    registered_artifact("results/reference/manuscript_reported_stage_contrasts.csv")
)
required_reference_columns = {
    "structure", "contrast", "context", "metric", "pair_scope",
    "median", "lower_95", "upper_95", "source_version",
}
if missing := sorted(required_reference_columns.difference(reference_targets.columns)):
    raise ValueError(f"Manuscript reference table is missing columns: {missing}")
if reference_targets.empty:
    raise ValueError("Manuscript reference table contains no contrast targets.")
reference_targets["analysis"] = reference_targets["structure"].map(normalize_analysis)
reference_targets["metric_canonical"] = reference_targets["metric"].astype(str).str.strip().str.lower()
reference_targets["estimand"] = reference_targets["contrast"].map({"stage": "delta", "stage_by_context": "psi"})
reference_targets["support"] = reference_targets["pair_scope"].map({
    "all_pairs": "all_pairs", "pairs_A_B": "common_support_A_B",
})
if reference_targets[["analysis", "estimand", "support"]].isna().any().any():
    raise ValueError("Reference table contains an unknown structure, contrast, or pair_scope.")
reference_key = ["analysis", "estimand", "context", "metric_canonical", "support"]
if reference_targets.duplicated(reference_key).any():
    raise ValueError("Reference table contains duplicate contrast targets.")
target_values = reference_targets[["median", "lower_95", "upper_95"]].apply(pd.to_numeric, errors="coerce")
if not np.isfinite(target_values.to_numpy()).all():
    raise ValueError("Reference table contains non-finite target intervals.")
validation_rows = []
for position, target in reference_targets.reset_index(drop=True).iterrows():
    analysis = target["analysis"]
    support = target["support"]
    estimand = target["estimand"]
    metric = target["metric_canonical"]
    context = normalize_context(target["context"]) if estimand == "delta" else None
    if estimand == "delta":
        source = combined_delta_summary if metric == "combined" else metric_delta_summary
    else:
        source = combined_psi_summary if metric == "combined" else metric_psi_summary
    observed = select_summary(source, analysis, support, metric, context)
    expected_median = target_values.loc[position, "median"]
    expected_lower = target_values.loc[position, "lower_95"]
    expected_upper = target_values.loc[position, "upper_95"]
    values_match = all(
        np.isclose(actual, expected, rtol=0, atol=0.015)
        for actual, expected in (
            (observed["estimate_median"], expected_median),
            (observed["lower_95"], expected_lower),
            (observed["upper_95"], expected_upper),
        )
    )
    validation_rows.append({
        "analysis": analysis, "support": support, "estimand": estimand,
        "metric": metric, "source_version": target["source_version"],
        "context": context or "difference", "expected_median": expected_median,
        "observed_median": observed["estimate_median"], "expected_lower_95": expected_lower,
        "observed_lower_95": observed["lower_95"], "expected_upper_95": expected_upper,
        "observed_upper_95": observed["upper_95"], "passes_at_2dp": values_match,
    })
headline_audit = pd.DataFrame(validation_rows)
display(headline_audit.round(3))
if not headline_audit["passes_at_2dp"].all():
    raise AssertionError("Saved posterior draws do not reproduce every registered manuscript contrast.")

## Validate checked-in summaries and diagnostics

The tabular exports are derived from the same saved draws. Diagnostics must show stable sampling, no divergent transitions, no maximum-tree-depth exceedances, and completed posterior-predictive and trace-plot artifacts.

In [ ]:
table_paths = {
    "stage effects by context and metric": "results/tables/stage_effects_by_context_metric.csv",
    "combined stage effects": "results/tables/stage_effects_combined.csv",
    "metric-specific psi": "results/tables/psi_by_metric.csv",
    "combined psi": "results/tables/psi_combined.csv",
    "model diagnostics": "results/tables/model_diagnostics.csv",
}
result_tables = {name: pd.read_csv(registered_artifact(path)) for name, path in table_paths.items()}
summary_columns = {
    "analysis", "support", "estimate_median", "estimate_mean", "lower_95", "upper_95",
    "Pr_effect_lt_0", "Pr_effect_gt_0", "n_draws", "n_pairs", "pairs_included",
}
for name, table in result_tables.items():
    if name == "model diagnostics":
        continue
    if missing := sorted(summary_columns.difference(table.columns)):
        raise ValueError(f"{name} is missing summary columns: {missing}")
    if not table["n_draws"].eq(8000).all():
        raise ValueError(f"{name} does not report 8,000 draws for every contrast.")

diagnostics = result_tables["model diagnostics"].copy()
diagnostic_columns = {
    "analysis", "max_rhat", "min_bulk_ess", "min_tail_ess",
    "n_divergent", "n_max_treedepth", "ppc_status", "ppc_path",
    "trace_path", "diagnostic_status",
}
if missing := sorted(diagnostic_columns.difference(diagnostics.columns)):
    raise ValueError(f"Model diagnostics are missing columns: {missing}")
if set(diagnostics["analysis"].map(normalize_analysis)) != {"sequence", "call"}:
    raise ValueError("Diagnostics must contain exactly the sequence and call models.")
if (diagnostics["max_rhat"] > 1.01).any():
    raise AssertionError("At least one model has max R-hat > 1.01.")
if (diagnostics[["min_bulk_ess", "min_tail_ess"]] < 1000).any().any():
    raise AssertionError("At least one minimum effective sample size is below 1,000.")
if diagnostics[["n_divergent", "n_max_treedepth"]].ne(0).any().any():
    raise AssertionError("Sampler diagnostics contain divergences or tree-depth exceedances.")
if not diagnostics["diagnostic_status"].astype(str).str.lower().isin({"pass", "passed", "ok"}).all():
    raise AssertionError("A model diagnostic status is not passing.")
if not diagnostics["ppc_status"].astype(str).str.lower().isin({"pass", "passed", "ok", "complete", "completed", "generated"}).all():
    raise AssertionError("A posterior-predictive check is not marked complete/passing.")
for column in ("ppc_path", "trace_path"):
    for recorded_path in diagnostics[column].astype(str):
        path = Path(recorded_path)
        if path.is_absolute():
            try:
                path = path.resolve().relative_to(ROOT)
            except ValueError as error:
                raise ValueError(f"Diagnostic path is outside the repository: {recorded_path}") from error
        registered_artifact(path.as_posix())

display(result_tables["combined stage effects"].round(3))
display(result_tables["combined psi"].round(3))
display(diagnostics)

## Figure 3

When a report-mode export generated from the validated draws is present, it is shown first. The locked manuscript-reference snapshot is shown otherwise. Both paths require checksum provenance; the notebook never substitutes a newly fitted model.

In [ ]:
regenerated = "results/figures/main/figure_3_regenerated.png"
locked = "results/figures/main/figure_3.png"
if (ROOT / regenerated).is_file():
    display("Figure 3 — regenerated from validated posterior draws",
            Image(filename=str(registered_artifact(regenerated))))
elif (ROOT / locked).is_file():
    print("A regenerated Figure 3 export is absent; showing the provenance-locked manuscript snapshot.")
    display(Image(filename=str(registered_artifact(locked))))
else:
    print("No optional Figure 3 PNG export is present. Posterior tables above remain fully validated.")

## Interpretation

The unified sequence model estimates a reduction in distance after pairing in the Non-partner context, but not in the Partner context; Ψ is positive, including on the two-pair common-support set. Three sequence metrics—transition probability, bigram, and local alignment—support the Non-partner reduction, whereas the repeat metric does not. The unified call model does not resolve a stage-related reduction in either context, and its Ψ interval includes zero. These statements come from the verified posterior draws summarized above.